In [2]:
import jax
import jax.numpy as jnp
from functools import partial
import numpy as np
from abc import ABC, abstractmethod

# ==============================================================================
# 1. ABSTRACT MANIFOLD BASE CLASS
# ==============================================================================
class Manifold(ABC):
    @property
    @abstractmethod
    def dim(self):
        pass

    @abstractmethod
    def identity(self):
        pass

    @abstractmethod
    def retract(self, point, delta_vec):
        """
        Pure Exponential map.
        MUST be differentiable at delta_vec=0 for gradients to work.
        """
        pass

    @abstractmethod
    def basis_projection(self, delta_vec):
        """Maps vector p -> Matrix X in Lie Algebra."""
        pass

# ==============================================================================
# 2. SU(3) MANIFOLD IMPLEMENTATION
# ==============================================================================
class SU3(Manifold):
    def __init__(self):
        self._basis = self._build_basis()

    @property
    def dim(self):
        return 8

    def identity(self):
        return jnp.eye(3, dtype=complex)

    def _build_basis(self):
        # Gell-Mann basis (Anti-Hermitian)
        λ = []
        λ.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]],dtype=complex))
        λ.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]],dtype=complex))
        λ.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]],dtype=complex))
        λ.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]],dtype=complex))
        λ.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]],dtype=complex))
        λ.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]],dtype=complex))
        λ.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]],dtype=complex))
        λ.append((1/jnp.sqrt(3))*jnp.array([[1,0,0],[0,1,0],[0,0,-2]],dtype=complex))

        B = jnp.stack(λ)
        return jnp.array([1j*b/jnp.sqrt(2) for b in B])

    def basis_projection(self, delta_vec):
        return jnp.tensordot(delta_vec, self._basis, axes=([0],[0]))

    def retract(self, point, delta_vec):
        # Pure exponential map (No clipping here to avoid NaN gradients)
        X = self.basis_projection(delta_vec)
        return jax.scipy.linalg.expm(X) @ point

# ==============================================================================
# 3. RICCATI OPTIMIZER ENGINE (FIXED)
# ==============================================================================
class RiccatiOptimizer:
    def __init__(self, manifold, step_size=0.1, sigma=0.1, stabilizer_type='riccati'):
        self.M = manifold
        self.eta = step_size
        self.sigma = sigma
        self.stabilizer_type = stabilizer_type

    @partial(jax.jit, static_argnums=(0, 1))
    def value_and_grad(self, objective_fn, point):
        """Computes function value and Riemannian gradient."""
        def local_f(coord):
            return objective_fn(self.M.retract(point, coord))

        val, g = jax.value_and_grad(local_f)(jnp.zeros(self.M.dim))
        # Ensure gradient is real to avoid complex warnings
        return val, jnp.real(g)

    def compute_hessian(self, objective_fn, point, grad_at_point, eps=1e-4):
        """Finite Difference Riemannian Hessian."""
        dim = self.M.dim
        H = jnp.zeros((dim, dim))

        def get_grad(p):
            _, g = self.value_and_grad(objective_fn, p)
            return g

        for i in range(dim):
            delta = jnp.zeros(dim).at[i].set(eps)
            point_eps = self.M.retract(point, delta)
            g_eps = get_grad(point_eps)
            # Explicit real cast to silence warnings
            col = jnp.real(g_eps - grad_at_point) / eps
            H = H.at[:, i].set(col)

        return 0.5 * (H + H.T)

    @partial(jax.jit, static_argnums=(0,))
    def _riccati_transform(self, eigenvalues):
        """Smooth Riccati Spectral Flow."""
        mu = jnp.sqrt(self.sigma / 2.0)
        T = jnp.tanh(self.eta * jnp.sqrt(2.0 * self.sigma))
        numer = eigenvalues + mu * T
        denom = 1.0 + (eigenvalues / mu) * T
        return numer / denom

    def step(self, objective_fn, point):
        """Performs one Stabilized Newton Step."""
        val, g = self.value_and_grad(objective_fn, point)
        H = self.compute_hessian(objective_fn, point, g)

        # Spectral Decomposition
        evals, evecs = jnp.linalg.eigh(H)

        if self.stabilizer_type == 'riccati':
            evals_stab = self._riccati_transform(evals)
        elif self.stabilizer_type == 'abs':
            evals_stab = jnp.abs(evals) + 1e-6
        else:
            evals_stab = evals

        # Solve for Newton Direction
        inv_evals = 1.0 / (evals_stab + 1e-8)
        H_inv = (evecs * inv_evals) @ evecs.T
        delta = -H_inv @ g

        # --- SAFE CLIPPING (Moved here to save the Gradient) ---
        # We clip the Update Vector, NOT the Manifold Chart
        nrm = jnp.linalg.norm(delta)
        # Clip max step norm to 0.5 to stay within chart validity
        scale = jnp.minimum(1.0, 0.5 / (nrm + 1e-12))
        delta = delta * scale

        new_point = self.M.retract(point, delta)
        return new_point, val

# ==============================================================================
# 4. DEMO RUNNER
# ==============================================================================
def run_demo():
    print("Initializing Fixed Riccati Optimizer...")

    su3 = SU3()
    opt = RiccatiOptimizer(su3, step_size=0.1, sigma=0.5, stabilizer_type='riccati')

    # Double Well Objective
    rng = np.random.default_rng(42)
    A1 = rng.normal(size=(3,3)) + 1j*rng.normal(size=(3,3)); A1 = A1/jnp.linalg.norm(A1)
    A2 = rng.normal(size=(3,3)) + 1j*rng.normal(size=(3,3)); A2 = A2/jnp.linalg.norm(A2)

    def loss_fn(U):
        e1 = -0.2 * jnp.real(jnp.trace(U.conj().T @ A1))
        e2 = -1.2 * jnp.real(jnp.trace(U.conj().T @ A2))
        UHU = U @ A1 @ U.conj().T
        diff = UHU - A1
        # Safe sum-sq norm
        saddle = 0.3 * jnp.sum(jnp.abs(diff)**2)
        return e1 + e2 + saddle

    U = su3.identity()
    print(f"{'Iter':<5} | {'Loss':<12} | {'Status'}")
    print("-" * 30)

    for i in range(15):
        U, val = opt.step(loss_fn, U)
        print(f"{i:<5} | {val:.6f}     | {'Stabilizing...' if i < 5 else 'Converging'}")

if __name__ == "__main__":
    run_demo()

Initializing Fixed Riccati Optimizer...
Iter  | Loss         | Status
------------------------------
0     | -0.854084     | Stabilizing...
1     | -1.108773     | Stabilizing...
2     | -1.181770     | Stabilizing...
3     | -1.183215     | Stabilizing...
4     | -1.183257     | Stabilizing...
5     | -1.183258     | Converging
6     | -1.183258     | Converging
7     | -1.183258     | Converging
8     | -1.183258     | Converging
9     | -1.183258     | Converging
10    | -1.183258     | Converging
11    | -1.183258     | Converging
12    | -1.183258     | Converging
13    | -1.183258     | Converging
14    | -1.183258     | Converging


In [5]:
import jax
import jax.numpy as jnp
import numpy as np
from functools import partial


# ================================================================
# 1. ABSTRACT MANIFOLD BASE CLASS
# ================================================================
class Manifold:
    @property
    def dim(self):
        raise NotImplementedError

    def identity(self):
        raise NotImplementedError

    def basis_projection(self, delta_vec):
        raise NotImplementedError

    def retract(self, point, delta_vec):
        raise NotImplementedError


# ================================================================
# 2. SO(3) MANIFOLD IMPLEMENTATION
# ================================================================
class SO3(Manifold):
    def __init__(self):
        self._basis = self._build_basis()

    @property
    def dim(self):
        return 3

    def identity(self):
        return jnp.eye(3)

    def _build_basis(self):
        """
        Standard skew-symmetric basis for so(3).
        """
        E1 = jnp.array([[0, 0, 0],
                        [0, 0, -1],
                        [0, 1, 0]], dtype=float)

        E2 = jnp.array([[0, 0, 1],
                        [0, 0, 0],
                        [-1, 0, 0]], dtype=float)

        E3 = jnp.array([[0, -1, 0],
                        [1,  0, 0],
                        [0,  0, 0]], dtype=float)

        return jnp.stack([E1, E2, E3], axis=0)

    def basis_projection(self, delta_vec):
        return jnp.tensordot(delta_vec, self._basis, axes=([0],[0]))

    def retract(self, point, delta_vec):
        """
        Exponential map on SO(3): R_new = exp(X) R.
        """
        X = self.basis_projection(delta_vec)
        return jax.scipy.linalg.expm(X) @ point


# ================================================================
# 3. RICCATI OPTIMIZER
# ================================================================
class RiccatiOptimizer:
    def __init__(self, manifold, step_size=0.15, sigma=0.4):
        self.M = manifold
        self.eta = step_size
        self.sigma = sigma

    @partial(jax.jit, static_argnums=(0,1))
    def value_and_grad(self, objective_fn, point):
        """
        Returns objective value and gradient in coordinate chart.
        """
        def local_f(coord):
            return objective_fn(self.M.retract(point, coord))

        val, g = jax.value_and_grad(local_f)(jnp.zeros(self.M.dim))
        return val, jnp.real(g)

    def compute_hessian(self, objective_fn, point, grad_at_point, eps=1e-4):
        """
        Finite difference Riemannian Hessian.
        """
        dim = self.M.dim
        H = jnp.zeros((dim, dim))

        def get_grad(p):
            _, g = self.value_and_grad(objective_fn, p)
            return g

        for i in range(dim):
            delta = jnp.zeros(dim).at[i].set(eps)
            p_eps = self.M.retract(point, delta)
            g_eps = get_grad(p_eps)
            H = H.at[:, i].set((g_eps - grad_at_point) / eps)

        return 0.5 * (H + H.T)

    @partial(jax.jit, static_argnums=(0,))
    def _riccati_transform(self, evals):
        """
        Smooth Riccati spectral flow.
        """
        mu = jnp.sqrt(self.sigma / 2.0)
        T = jnp.tanh(self.eta * jnp.sqrt(2.0 * self.sigma))
        return (evals + mu*T) / (1 + (evals/mu)*T)

    def step(self, objective_fn, point):
        """
        Performs a Riccati-stabilized Newton step.
        """
        val, g = self.value_and_grad(objective_fn, point)
        H = self.compute_hessian(objective_fn, point, g)

        evals, evecs = jnp.linalg.eigh(H)
        evals_stab = self._riccati_transform(evals)

        inv_evals = 1.0 / (evals_stab + 1e-8)
        H_inv = (evecs * inv_evals) @ evecs.T

        delta_vec = -H_inv @ g

        # Clip step in coordinate space
        nrm = jnp.linalg.norm(delta_vec)
        delta_vec = delta_vec * jnp.minimum(1.0, 0.5/(nrm + 1e-12))

        return self.M.retract(point, delta_vec), val


# ================================================================
# 4. **CORRECTED** SO(3) SADDLE OBJECTIVE (ASYMMETRIC, TRUE SADDLE)
# ================================================================
def so3_saddle_objective(R):
    """
    A guaranteed SO(3) saddle:
      - Asymmetric linear term (break R -> -R symmetry)
      - Symmetric matrix with distinct eigenvalues
      - Produces real curvature variation and descent directions
    """

    # Break symmetry with a non-symmetric target
    A = jnp.array([[1.0,  0.3, -0.2],
                   [0.1, -0.5,  0.4],
                   [0.2,  0.0,  0.7]], dtype=float)

    # Distinct eigenvalues create curvature structure
    N = jnp.array([[2.0, 0.1, 0.0],
                   [0.1, -1.0, 0.3],
                   [0.0, 0.3, 0.5]], dtype=float)

    # Directional pull (primary well)
    pull = -1.2 * jnp.trace(R.T @ A)

    # Saddle term
    RNR = R @ N @ R.T
    diff = RNR - N
    saddle = 0.75 * jnp.sum(diff * diff)

    return pull + saddle


# ================================================================
# 5. DEMO RUNNER
# ================================================================
def run_so3_demo():
    so3 = SO3()
    opt = RiccatiOptimizer(so3, step_size=0.15, sigma=0.35)

    R = so3.identity()

    print("Iter | Loss")
    print("-------------")

    for i in range(20):
        R, val = opt.step(so3_saddle_objective, R)
        print(f"{i:02d}   | {float(val): .6f}")


if __name__ == "__main__":
    run_so3_demo()


Iter | Loss
-------------
00   | -1.440000
01   | -1.195358
02   | -0.088337
03   | -0.967629
04   | -0.111958
05   | -0.966420
06   | -0.112016
07   | -0.966429
08   | -0.112014
09   | -0.966428
10   | -0.112014
11   | -0.966429
12   | -0.112014
13   | -0.966430
14   | -0.112014
15   | -0.966429
16   | -0.112015
17   | -0.966429
18   | -0.112014
19   | -0.966430


In [6]:
import jax
import jax.numpy as jnp
# FIX 1: Explicitly import jax.scipy to access linalg.expm
import jax.scipy.linalg
import numpy as np
from functools import partial

# Enable 64-bit precision for stable geometric ops
jax.config.update("jax_enable_x64", True)

# ================================================================
# 1. ABSTRACT MANIFOLD BASE CLASS
# ================================================================
class Manifold:
    @property
    def dim(self):
        raise NotImplementedError

    def identity(self):
        raise NotImplementedError

    def basis_projection(self, delta_vec):
        raise NotImplementedError

    def retract(self, point, delta_vec):
        raise NotImplementedError


# ================================================================
# 2. SO(3) MANIFOLD IMPLEMENTATION
# ================================================================
class SO3(Manifold):
    def __init__(self):
        self._basis = self._build_basis()

    @property
    def dim(self):
        return 3

    def identity(self):
        return jnp.eye(3)

    def _build_basis(self):
        """
        Standard skew-symmetric basis for so(3).
        """
        E1 = jnp.array([[0, 0, 0],
                        [0, 0, -1],
                        [0, 1, 0]], dtype=float)

        E2 = jnp.array([[0, 0, 1],
                        [0, 0, 0],
                        [-1, 0, 0]], dtype=float)

        E3 = jnp.array([[0, -1, 0],
                        [1,  0, 0],
                        [0,  0, 0]], dtype=float)

        return jnp.stack([E1, E2, E3], axis=0)

    def basis_projection(self, delta_vec):
        return jnp.tensordot(delta_vec, self._basis, axes=([0],[0]))

    def retract(self, point, delta_vec):
        """
        Exponential map on SO(3): R_new = exp(X) R
        """
        X = self.basis_projection(delta_vec)
        # FIX 1: Correct usage of jax.scipy.linalg
        return jax.scipy.linalg.expm(X) @ point


# ================================================================
# 3. RICCATI OPTIMIZER
# ================================================================
class RiccatiOptimizer:
    def __init__(self, manifold, step_size=0.15, sigma=0.4):
        self.M = manifold
        self.eta = step_size
        self.sigma = sigma

    @partial(jax.jit, static_argnums=(0,1))
    def value_and_grad(self, objective_fn, point):
        """
        Returns objective value and gradient in coordinate chart.
        """
        def local_f(coord):
            return objective_fn(self.M.retract(point, coord))

        val, g = jax.value_and_grad(local_f)(jnp.zeros(self.M.dim))
        return val, jnp.real(g)

    def compute_hessian(self, objective_fn, point, grad_at_point, eps=1e-4):
        """
        Finite difference Riemannian Hessian.
        """
        dim = self.M.dim
        H = jnp.zeros((dim, dim))

        def get_grad(p):
            _, g = self.value_and_grad(objective_fn, p)
            return g

        for i in range(dim):
            delta = jnp.zeros(dim).at[i].set(eps)
            p_eps = self.M.retract(point, delta)
            g_eps = get_grad(p_eps)
            H = H.at[:, i].set((g_eps - grad_at_point) / eps)

        return 0.5 * (H + H.T)

    @partial(jax.jit, static_argnums=(0,))
    def _riccati_transform(self, evals):
        """
        Smooth Riccati spectral flow.
        FIX 2: Stabilized denominator to prevent singularities at poles.
        """
        mu = jnp.sqrt(self.sigma / 2.0)
        T = jnp.tanh(self.eta * jnp.sqrt(2.0 * self.sigma))

        numerator = evals + mu*T
        denominator = 1.0 + (evals/mu)*T

        # Safe division: if denominator is too close to 0, fallback or clamp
        # Here we just add a tiny epsilon for safety in this demo
        safe_denom = jnp.where(jnp.abs(denominator) < 1e-6, 1e-6, denominator)

        return numerator / safe_denom

    def step(self, objective_fn, point):
        """
        Performs a Riccati-stabilized Newton step.
        """
        val, g = self.value_and_grad(objective_fn, point)
        H = self.compute_hessian(objective_fn, point, g)

        evals, evecs = jnp.linalg.eigh(H)
        evals_stab = self._riccati_transform(evals)

        inv_evals = 1.0 / (evals_stab + 1e-8)
        H_inv = (evecs * inv_evals) @ evecs.T

        delta_vec = -H_inv @ g

        # Clip step in coordinate space (safe, differentiable)
        nrm = jnp.linalg.norm(delta_vec)
        delta_vec = delta_vec * jnp.minimum(1.0, 0.5 / (nrm + 1e-12))

        return self.M.retract(point, delta_vec), val


# ================================================================
# 4. SO(3) SADDLE OBJECTIVE
# ================================================================
def so3_saddle_objective(R):
    """
    Guaranteed SO(3) saddle structure.
    """
    # Rotation pull component (skew-symmetric)
    M = jnp.array([[0, -1,  0],
                   [1,  0,  0],
                   [0,  0,  0]], dtype=float)

    # Distinct eigenvalue structure (saddle)
    N = jnp.diag(jnp.array([1.0, -2.0, 0.5]))

    # Well 1: directional pull
    well = -0.9 * jnp.trace(R.T @ M)

    # Saddle term
    RNR = R @ N @ R.T
    diff = RNR - N
    saddle = 0.6 * jnp.sum(diff * diff)

    return well + saddle


# ================================================================
# 5. DEMO RUNNER
# ================================================================
def run_so3_demo():
    so3 = SO3()
    # Initialize near Identity
    opt = RiccatiOptimizer(so3, step_size=0.20, sigma=0.35)

    R = so3.identity()

    print(f"{'Iter':<5} | {'Loss':<12}")
    print("-" * 20)

    for i in range(20):
        R, val = opt.step(so3_saddle_objective, R)
        print(f"{i:02d}    | {float(val): .6f}")


if __name__ == "__main__":
    run_so3_demo()

Iter  | Loss        
--------------------
00    |  0.000000
01    |  1.619402
02    | -0.000000
03    |  1.619402
04    | -0.000000
05    |  1.619402
06    | -0.000000
07    |  1.619402
08    | -0.000000
09    |  1.619402
10    | -0.000000
11    |  1.619402
12    | -0.000000
13    |  1.619402
14    | -0.000000
15    |  1.619402
16    | -0.000000
17    |  1.619402
18    | -0.000000
19    |  1.619402


In [7]:
import jax
import jax.numpy as jnp
import jax.scipy.linalg  # Explicit import needed for expm
import numpy as np
from functools import partial

# Enable 64-bit precision
jax.config.update("jax_enable_x64", True)

# ================================================================
# 1. ABSTRACT MANIFOLD BASE CLASS
# ================================================================
class Manifold:
    @property
    def dim(self):
        raise NotImplementedError

    def identity(self):
        raise NotImplementedError

    def basis_projection(self, delta_vec):
        raise NotImplementedError

    def retract(self, point, delta_vec):
        raise NotImplementedError


# ================================================================
# 2. SO(3) MANIFOLD IMPLEMENTATION
# ================================================================
class SO3(Manifold):
    def __init__(self):
        self._basis = self._build_basis()

    @property
    def dim(self):
        return 3

    def identity(self):
        return jnp.eye(3)

    def _build_basis(self):
        # Standard skew-symmetric basis
        E1 = jnp.array([[0, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=float)
        E2 = jnp.array([[0, 0, 1], [0, 0, 0], [-1, 0, 0]], dtype=float)
        E3 = jnp.array([[0, -1, 0], [1,  0, 0], [0,  0, 0]], dtype=float)
        return jnp.stack([E1, E2, E3], axis=0)

    def basis_projection(self, delta_vec):
        return jnp.tensordot(delta_vec, self._basis, axes=([0],[0]))

    def retract(self, point, delta_vec):
        X = self.basis_projection(delta_vec)
        return jax.scipy.linalg.expm(X) @ point


# ================================================================
# 3. RICCATI OPTIMIZER (FIXED)
# ================================================================
class RiccatiOptimizer:
    def __init__(self, manifold, step_size=0.15, sigma=0.4):
        self.M = manifold
        self.eta = step_size  # This is your Learning Rate
        self.sigma = sigma

    @partial(jax.jit, static_argnums=(0,1))
    def value_and_grad(self, objective_fn, point):
        def local_f(coord):
            return objective_fn(self.M.retract(point, coord))
        val, g = jax.value_and_grad(local_f)(jnp.zeros(self.M.dim))
        return val, jnp.real(g)

    def compute_hessian(self, objective_fn, point, grad_at_point, eps=1e-4):
        dim = self.M.dim
        H = jnp.zeros((dim, dim))

        def get_grad(p):
            _, g = self.value_and_grad(objective_fn, p)
            return g

        for i in range(dim):
            delta = jnp.zeros(dim).at[i].set(eps)
            p_eps = self.M.retract(point, delta)
            g_eps = get_grad(p_eps)
            H = H.at[:, i].set((g_eps - grad_at_point) / eps)

        return 0.5 * (H + H.T)

    @partial(jax.jit, static_argnums=(0,))
    def _riccati_transform(self, evals):
        # Stabilization logic
        mu = jnp.sqrt(self.sigma / 2.0)
        T = jnp.tanh(self.eta * jnp.sqrt(2.0 * self.sigma))

        numerator = evals + mu*T
        denominator = 1.0 + (evals/mu)*T

        # Safety check to prevent division by zero
        safe_denom = jnp.where(jnp.abs(denominator) < 1e-6, 1e-6, denominator)
        return numerator / safe_denom

    def step(self, objective_fn, point):
        val, g = self.value_and_grad(objective_fn, point)
        H = self.compute_hessian(objective_fn, point, g)

        evals, evecs = jnp.linalg.eigh(H)
        evals_stab = self._riccati_transform(evals)

        inv_evals = 1.0 / (evals_stab + 1e-8)
        H_inv = (evecs * inv_evals) @ evecs.T

        # Calculate Newton direction
        delta_vec = -H_inv @ g

        # CRITICAL FIX: Apply step size (Learning Rate)
        delta_vec = delta_vec * self.eta

        # Safety Clipping (0.5 radius)
        nrm = jnp.linalg.norm(delta_vec)
        scale = jnp.minimum(1.0, 0.5 / (nrm + 1e-12))
        delta_vec = delta_vec * scale

        return self.M.retract(point, delta_vec), val


# ================================================================
# 4. SO(3) SADDLE OBJECTIVE
# ================================================================
def so3_saddle_objective(R):
    # Asymmetric target
    A = jnp.array([[1.0,  0.3, -0.2],
                   [0.1, -0.5,  0.4],
                   [0.2,  0.0,  0.7]], dtype=float)

    # Eigenvalue structure
    N = jnp.array([[2.0, 0.1, 0.0],
                   [0.1, -1.0, 0.3],
                   [0.0, 0.3, 0.5]], dtype=float)

    pull = -1.2 * jnp.trace(R.T @ A)

    RNR = R @ N @ R.T
    diff = RNR - N
    saddle = 0.75 * jnp.sum(diff * diff)

    return pull + saddle


# ================================================================
# 5. DEMO RUNNER
# ================================================================
def run_so3_demo():
    so3 = SO3()
    # Reduced step size to 0.10 to prevent bouncing
    opt = RiccatiOptimizer(so3, step_size=0.10, sigma=0.4)

    R = so3.identity()

    print(f"{'Iter':<5} | {'Loss':<12}")
    print("-" * 20)

    for i in range(25):
        R, val = opt.step(so3_saddle_objective, R)
        print(f"{i:02d}    | {float(val): .6f}")


if __name__ == "__main__":
    run_so3_demo()

Iter  | Loss        
--------------------
00    | -1.440000
01    | -1.453272
02    | -1.459854
03    | -1.463391
04    | -1.465324
05    | -1.466384
06    | -1.466966
07    | -1.467285
08    | -1.467461
09    | -1.467557
10    | -1.467610
11    | -1.467639
12    | -1.467655
13    | -1.467663
14    | -1.467668
15    | -1.467671
16    | -1.467672
17    | -1.467673
18    | -1.467673
19    | -1.467674
20    | -1.467674
21    | -1.467674
22    | -1.467674
23    | -1.467674
24    | -1.467674


In [8]:
import jax
import jax.numpy as jnp
import jax.scipy.linalg
import numpy as np
from functools import partial

jax.config.update("jax_enable_x64", True)

# ================================================================
# 1. ABSTRACT MANIFOLD BASE CLASS
# ================================================================
class Manifold:
    @property
    def dim(self):
        raise NotImplementedError
    def identity(self):
        raise NotImplementedError
    def basis_projection(self, delta_vec):
        raise NotImplementedError
    def retract(self, point, delta_vec):
        raise NotImplementedError


# ================================================================
# 2. SO(3) MANIFOLD
# ================================================================
class SO3(Manifold):
    def __init__(self):
        self._basis = self._build_basis()

    @property
    def dim(self):
        return 3

    def identity(self):
        return jnp.eye(3)

    def _build_basis(self):
        E1 = jnp.array([[0, 0, 0],
                        [0, 0, -1],
                        [0, 1, 0]], dtype=float)

        E2 = jnp.array([[0, 0, 1],
                        [0, 0, 0],
                        [-1, 0, 0]], dtype=float)

        E3 = jnp.array([[0, -1, 0],
                        [1,  0, 0],
                        [0,  0, 0]], dtype=float)

        return jnp.stack([E1, E2, E3], axis=0)

    def basis_projection(self, delta_vec):
        return jnp.tensordot(delta_vec, self._basis, axes=([0],[0]))

    def retract(self, point, delta_vec):
        X = self.basis_projection(delta_vec)
        return jax.scipy.linalg.expm(X) @ point


# ================================================================
# 3. PURE NEWTON OPTIMIZER (NO Stabilization)
# ================================================================
class NewtonOptimizer:
    def __init__(self, manifold, step_size=1.0):
        self.M = manifold
        self.eta = step_size

    @partial(jax.jit, static_argnums=(0,1))
    def value_and_grad(self, objective_fn, point):
        def local_f(coord):
            return objective_fn(self.M.retract(point, coord))
        val, g = jax.value_and_grad(local_f)(jnp.zeros(self.M.dim))
        return val, jnp.real(g)

    def compute_hessian(self, objective_fn, point, grad_at_point, eps=1e-4):
        dim = self.M.dim
        H = jnp.zeros((dim, dim))

        def get_grad(p):
            _, g = self.value_and_grad(objective_fn, p)
            return g

        for i in range(dim):
            delta = jnp.zeros(dim).at[i].set(eps)
            p_eps = self.M.retract(point, delta)
            g_eps = get_grad(p_eps)
            H = H.at[:, i].set((g_eps - grad_at_point) / eps)

        return 0.5 * (H + H.T)

    def step(self, objective_fn, point):
        val, g = self.value_and_grad(objective_fn, point)
        H = self.compute_hessian(objective_fn, point, g)

        # --- UNSTABLE STEP: INVERSE OF RAW HESSIAN ---
        evals, evecs = jnp.linalg.eigh(H)
        inv_evals = 1.0 / (evals + 1e-12)  # <-- NO STABILIZATION
        H_inv = (evecs * inv_evals) @ evecs.T

        delta_vec = -H_inv @ g
        new_point = self.M.retract(point, delta_vec)
        return new_point, val


# ================================================================
# 4. SAME SO(3) SADDLE OBJECTIVE AS RICCATI DEMO
# ================================================================
def so3_saddle_objective(R):
    A = jnp.array([[1.0,  0.3, -0.2],
                   [0.1, -0.5,  0.4],
                   [0.2,  0.0,  0.7]], dtype=float)

    N = jnp.array([[2.0, 0.1, 0.0],
                   [0.1, -1.0, 0.3],
                   [0.0, 0.3, 0.5]], dtype=float)

    pull = -1.2 * jnp.trace(R.T @ A)
    RNR = R @ N @ R.T
    diff = RNR - N
    saddle = 0.75 * jnp.sum(diff * diff)
    return pull + saddle


# ================================================================
# 5. GO TIME: RUN PURE NEWTON ON THE SADDLE
# ================================================================
def run_newton_failure_demo():
    so3 = SO3()
    opt = NewtonOptimizer(so3, step_size=1.0)

    R = so3.identity()

    print(f"{'Iter':<5} | {'Loss':<12}")
    print("-" * 25)

    for i in range(20):
        R, val = opt.step(so3_saddle_objective, R)
        print(f"{i:02d}    | {float(val): .6f}")


if __name__ == "__main__":
    run_newton_failure_demo()


Iter  | Loss        
-------------------------
00    | -1.440000
01    | -1.467674
02    | -1.467674
03    | -1.467674
04    | -1.467674
05    | -1.467674
06    | -1.467674
07    | -1.467674
08    | -1.467674
09    | -1.467674
10    | -1.467674
11    | -1.467674
12    | -1.467674
13    | -1.467674
14    | -1.467674
15    | -1.467674
16    | -1.467674
17    | -1.467674
18    | -1.467674
19    | -1.467674
